# Stage 4: the size rule

**Reads:** `results/dev_runs.csv` and `results/dev_components.csv`, saved by the full run of stage 3.

**Writes:**
- `results/size_rule.csv`: at each training size, `ppml`, tuned `rf` and tuned `gbm`, with each one's setting, mean test deviance, standard error and median, whether one draw holds more than half of the sum over draws (`se_unreliable`), the one the size rule chooses, and the settings that sit at an edge of their grid. Stage 5 reads the settings from it.
- `results/gaps.csv`: at each size, the error, bias and variance gaps of `ppml` against tuned `rf`, tuned `gbm` and the better of the two (`best`), with the draws in which `ppml` has the lower test deviance.
- `results/cutoff.csv`: for each of the three comparisons, the pattern of the error gap over the sizes, its n* or `last_crossing` with the two sizes around it (the n* of `best` is the development-year n*), the share of bootstrap resamples in each pattern, and the bootstrap intervals.

The steps are those of stage 4 in section 6 of `docs/build_plan.md`, with step 0, the rule Stability on the sizes the checkpoint did not see, first. The stage fits nothing. Everything here is the development year, which is open to exploration: the ensembles are tuned on the same test set they are scored on, which favours them, so the development-year n* is exploratory. H1 is tested in stage 5, on the analysis year, at the settings frozen at the end of this stage.

Every standard error and interval here describes variation over training draws from one pool, scored on one test set. The draws overlap, more so at large sizes.

*Drafted with Claude (Anthropic), which was asked for stage 4 of the build plan: the rule Stability on the sizes above `CHECKPOINT_MAX`, the scores and tuning, the size rule, the gaps, the cutoff with its bootstrap interval, and the freeze. It replaces the placeholder of 18 October 2026.*

In [ ]:
from helpers import (BOOTSTRAP, CHECKPOINT_MAX, DEV_YEAR, DRAWS, PATTERNS, RESULTS, ROOT, SIZES,
                     better_ensemble, bootstrap_crossings, candidate_matrices, check_find_crossing, check_resampling,
                     crossings, cutoff_table, divergences, gaps, gaps_with_wins, gbm_extremes, line_configs, per_draw,
                     ppml_wins, refuse_part, size_rule_table, summarise, tune)

import numpy as np
import pandas as pd

refuse_part()
# Section 8: find_crossing on toy gaps, before it is used on real ones.
check_find_crossing()

VARIANT, SPLIT = "base", 0
# round_trip reads every number back exactly as stage 3 wrote it.
runs = pd.read_csv(RESULTS / "dev_runs.csv", float_precision="round_trip")
components = pd.read_csv(RESULTS / "dev_components.csv", float_precision="round_trip")
sizes = sorted(int(n) for n in runs["size"].unique())
print(f"Read {len(runs):,} rows of dev_runs.csv and {len(components):,} rows of dev_components.csv")
print(f"Sizes: {sizes}; {runs['draw'].nunique()} draws; {runs['config'].nunique()} configurations")

## Before the steps: the inputs are complete

The files hold the base experiment of the development year only, with every size of `SIZES` and `DRAWS` draws for every size and configuration; every test deviance is finite; and `dev_components.csv` has one row for each size and configuration. The files of part 1, for example, fail the check of the sizes: they lack the sizes above `CHECKPOINT_MAX`. Stage 3 checks that every prediction is positive and finite and writes no file if one is not, so a failure here means the inputs are not the files of the full run of stage 3.

In [ ]:
for table in (runs, components):
    assert (table["variant"] == VARIANT).all() and (table["year"] == DEV_YEAR).all() and (table["split"] == SPLIT).all(), \
        "The inputs hold rows other than the base experiment of the development year."
assert sizes == list(SIZES), f"The inputs have the sizes {sizes}, not SIZES {list(SIZES)}: they are not the files of the full run of stage 3."
draws_per = runs.groupby(["size", "config"])["draw"].nunique()
assert draws_per.nunique() == 1, "Some size and configuration has a different number of draws."
assert draws_per.iloc[0] == DRAWS, f"The inputs have {draws_per.iloc[0]} draws per size, not DRAWS {DRAWS}."
assert np.isfinite(runs["test_deviance"]).all(), "A test deviance in the inputs is not finite."
assert len(components) == len(draws_per), "dev_components.csv does not have one row per size and configuration."
print(f"{draws_per.iloc[0]} draws for each of {len(draws_per):,} sizes and configurations; every test deviance is finite")

## Step 0. The rule Stability on the sizes above `CHECKPOINT_MAX`

The checkpoint looked for diverging boosted models on the sizes of part 1 only. The rule Stability (section 6) asks for the same test on the larger sizes after the full run of stage 3, before stage 4 uses its files. This step runs it, before any other: a `gbm` configuration on a draw diverges if its test deviance is more than ten times that of `mean` on the same sample, or if its largest test prediction is more than 1,000 times the largest training flow (`divergences`, as in decision 9). If one does, the stage stops here and writes nothing, and Claude reports it to Yu. The rule then raises `GBM_MIN_LEAF` from 20 to 50, a change Yu confirms as a new row of section 2, and stage 3 is run again in full. The full run of stage 3 checks every row of part 1 against the rerun of part 1 in `results/checkpoint/rerun/inputs/`, and every `gbm` row would differ, so the same row of section 2 also says how that check is handled.

For each of these sizes, the first table shows how close the boosted models came to each threshold: the largest test deviance of a `gbm` configuration divided by that of `mean` on the same draw, and the largest test prediction divided by the largest training flow. If a configuration on a draw crosses a threshold, a second table names it.

In [ ]:
above = runs[runs["size"] > CHECKPOINT_MAX]
stability = gbm_extremes(above)
diverged = divergences(above)
print(f"Sizes above CHECKPOINT_MAX ({CHECKPOINT_MAX:,}):")
print(stability.to_string(index=False, float_format="{:.3g}".format))
if len(diverged) > 0:
    print()
    print("The gbm configurations and draws that cross a threshold:")
    print(diverged.to_string(index=False))
assert len(diverged) == 0, (
    "A gbm draw diverges above CHECKPOINT_MAX, so the rule Stability applies (section 6). Stop and report to Yu: "
    "GBM_MIN_LEAF goes from 20 to 50 in a new row of section 2, which also says how the full run of stage 3 is "
    "checked against part 1, and stage 3 is run again in full.")
print("No gbm draw diverges: the rule Stability does not fire.")

## Steps 1 to 3. Scores, tuning and the size rule

- Step 1: for every size and configuration, the mean test deviance over draws with its standard error (the standard deviation over draws divided by the square root of the number of draws) and the median (`summarise`). Where one draw holds more than half of the sum over draws, the standard error is marked unreliable.
- Step 2: the tuned setting of `rf` and of `gbm` at each size is the one with the lowest mean test deviance (`tune`). A tie goes to the simpler setting: for `gbm` the smaller depth, then fewer trees; for `rf` the larger `min_samples_leaf`, then fewer predictors per split.
- Step 3: at each size, the size rule chooses the one of `ppml`, tuned `rf` and tuned `gbm` with the lowest mean test deviance (`size_rule`; a tie goes to the first of the three). The mean decides (decision 12); the median is there to be read.

`size_rule.csv` has these three rows for every size. For `rf` and `gbm`, the column `edges` names any setting that sits at an edge of its grid, as item 5 of the checkpoint defines it. The rule Grid edges had its one round at the checkpoint (decision 19), so such an edge is reported, not extended: it is a limit of the grid, or a side that cannot be extended.

In [ ]:
summary = summarise(runs)
tuned = tune(summary)
rule = size_rule_table(summary, tuned, runs)
with pd.option_context("display.width", 200, "display.max_columns", None, "display.max_colwidth", 80):
    print(rule.to_string(index=False, float_format="{:.4g}".format))
print()
print("The model the size rule chooses at each size:")
print(rule.loc[rule["chosen"], ["size", "model", "config"]].to_string(index=False))

## Step 4. Gaps

For `ppml` against tuned `rf`, against tuned `gbm` and against the better of the two (`best`), at each size:
- error gap = E(ppml) - E(c), with its standard error paired over draws, and whether `ppml` is within two standard errors of the ensemble (the error gap at most twice its standard error). A negative error gap means `ppml` has the lower mean test deviance;
- bias gap = B(ppml) - B(c) and variance gap = V(c) - V(ppml), in the three versions of section 7 (raw, r and rn), with the check error gap = bias gap - variance gap (`gaps` stops if it fails);
- the number of draws in which `ppml` has the lower test deviance than that ensemble on the same training sample (decision 12; a tie is not a win).

In [ ]:
lines = line_configs(tuned)
wins = ppml_wins(per_draw(runs, lines), better_ensemble(tuned))
gap_table = gaps_with_wins(gaps(components, runs, tuned), wins)
print(gap_table[["size", "comparison", "config", "error_gap", "se_error_gap", "within_2se", "ppml_wins", "draws",
                 "bias_gap_rn", "variance_gap_rn"]].to_string(index=False, float_format="{:.4g}".format))

## Step 5. The cutoff

`find_crossing` reads the error gap of each comparison over the sizes (section 7): `one_crossing` gives n*, interpolated in log n between the two sizes around the change; `reversal_then_crossing` gives `last_crossing` the same way (decision 12). The `best` comparison gives the development-year n*. It is exploratory: the ensembles are tuned on this test set, which favours them and moves n* down, and the checkpoint changed the design after part 1 was seen. The test of H1 is `analysis_cutoff.csv` in stage 5.

In [ ]:
crossing_table = crossings(gap_table)
print(crossing_table.to_string(index=False, float_format="{:.1f}".format))

## Step 6. The bootstrap interval

The draws are resampled with replacement `BOOTSTRAP` times, separately at each size, with the same resampled draws for every configuration (resample b at size n uses `rng("bootstrap", year, split, n, b)`). Inside each resample, `rf` and `gbm` are tuned again, the error gaps recomputed and `find_crossing` read again. The bias and variance gaps are not resampled: they need each draw's predictions, which stage 3 does not save.

First, the same code is run on every draw once, without resampling: it must give the tuned settings of step 2 and, to a relative 1e-9, the error gaps of step 4 (`check_resampling`). Then, for each comparison, the share of resamples in each pattern, the 2.5 and 97.5 percentiles of n* over the resamples with `one_crossing`, and of `last_crossing` over the resamples with `reversal_then_crossing`, each with the number of resamples it rests on.

In [ ]:
matrices = candidate_matrices(runs)
check_resampling(matrices, tuned, gap_table)
print("On every draw once, the bootstrap gives the tuned settings of step 2, and the error gaps of step 4 to a relative 1e-9.")
resampled = bootstrap_crossings(matrices, DEV_YEAR, SPLIT, BOOTSTRAP)
cutoff = cutoff_table(crossing_table, resampled)
print(f"{BOOTSTRAP:,} resamples")
print()
print("Share of resamples in each pattern:")
print(cutoff[["comparison"] + [f"share_{pattern}" for pattern in PATTERNS]].to_string(index=False, float_format="{:.3f}".format))
print()
print(cutoff[["comparison", "pattern", "n_star", "resamples_one_crossing", "n_star_low", "n_star_high",
              "last_crossing", "resamples_reversal_then_crossing", "last_crossing_low", "last_crossing_high"]]
      .to_string(index=False, float_format="{:.1f}".format))

## Write the three files

In [ ]:
RESULTS.mkdir(exist_ok=True)
for table, name in [(rule, "size_rule.csv"), (gap_table, "gaps.csv"), (cutoff, "cutoff.csv")]:
    table.to_csv(RESULTS / name, index=False, lineterminator="\n")
    print(f"Wrote {(RESULTS / name).relative_to(ROOT)}: {len(table):,} rows")

## Step 7. The freeze

Stage 4 ends with the second pause of the plan. Claude posts the tuned settings, the size rule and the development-year n* below in the plan thread, and Yu confirms them; the confirmation goes into section 2 of the plan as a new row, with the commit of `results/size_rule.csv`. From then on nothing is tuned, and stage 5 reads the analysis year for the first time, at the settings in `size_rule.csv`.

In [ ]:
print("Tuned settings and the size rule, by size:")
frozen = rule.pivot(index="size", columns="model", values="config")[["ppml", "rf", "gbm"]]
# Both are indexed by size, so each size gets the model chosen at it.
frozen["size_rule_chooses"] = rule.loc[rule["chosen"]].set_index("size")["model"]
print(frozen.to_string())
print()
best = cutoff.set_index("comparison").loc["best"]
print(f"Development-year cutoff, best comparison: {best['pattern']}")
if best["pattern"] == "one_crossing":
    print(f"  n* = {best['n_star']:,.0f}, between {best['crossing_from']:,} and {best['crossing_to']:,}")
if best["pattern"] == "reversal_then_crossing":
    print(f"  last_crossing = {best['last_crossing']:,.0f}, between {best['crossing_from']:,} and {best['crossing_to']:,}")
shares = ", ".join(f"{pattern} {best[f'share_{pattern}']:.1%}" for pattern in PATTERNS)
print(f"  bootstrap, {best['resamples']:,} resamples: {shares}")
if best["resamples_one_crossing"] > 0:
    print(f"  n* over the {best['resamples_one_crossing']:,} resamples with one_crossing: "
          f"{best['n_star_low']:,.0f} to {best['n_star_high']:,.0f}")
if best["resamples_reversal_then_crossing"] > 0:
    print(f"  last_crossing over the {best['resamples_reversal_then_crossing']:,} resamples with reversal_then_crossing: "
          f"{best['last_crossing_low']:,.0f} to {best['last_crossing_high']:,.0f}")